# 1. Tiêu đề / giới thiệu

Dự đoán giá nhà bằng KNN Regression

Notebook này hoàn thành Bước 2: xây dựng Machine Learning KNN Regression trên Google Colab cho bài toán dự đoán giá nhà. Phạm vi chỉ gồm dataset, train/test split, StandardScaler, KNeighborsRegressor, thử nhiều giá trị K, đánh giá, save/load model và predict thử.

## 2. Cài đặt thư viện

Cell này kiểm tra và cài các thư viện Machine Learning cần thiết nếu môi trường Colab chưa có sẵn. Không cài FastAPI, Uvicorn hoặc pyngrok trong Bước 2.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}

missing_packages = [
    package_name
    for module_name, package_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("Tất cả thư viện Machine Learning cần thiết đã được cài sẵn.")

## 3. Import thư viện

Import các thư viện cần dùng cho xử lý dữ liệu, train model, đánh giá metrics và lưu model.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 4. Tạo dataset

Tạo synthetic dataset gồm area, rooms, distance và price. Price được tạo theo quan hệ gần tuyến tính có Gaussian noise để dữ liệu không hoàn hảo tuyệt đối.

In [ ]:
random_seed = 42
n_samples = 1000
rng = np.random.default_rng(random_seed)

area = rng.uniform(30, 250, n_samples)
rooms = rng.integers(1, 7, n_samples)
distance = rng.uniform(0.5, 30, n_samples)
noise = rng.normal(0, 150, n_samples)

price = 500 + 18 * area + 220 * rooms - 30 * distance + noise
price = np.maximum(price, 300)

df = pd.DataFrame(
    {
        "area": np.round(area, 2),
        "rooms": rooms,
        "distance": np.round(distance, 2),
        "price": np.round(price, 2),
    }
)

print(f"Đã tạo dataset gồm {len(df)} mẫu dữ liệu.")

## 5. Khám phá dữ liệu

Kiểm tra nhanh 5 dòng đầu, kích thước dữ liệu, thông tin cột, thống kê mô tả, missing values và kiểu dữ liệu.

In [ ]:
display_columns_vi = {
    "area": "Diện tích (m2)",
    "rooms": "Số phòng",
    "distance": "Khoảng cách tới trung tâm (km)",
    "price": "Giá nhà (triệu VND)",
}

print("5 dòng đầu tiên của dataset:")
display(df.head().rename(columns=display_columns_vi))

print(f"Kích thước dataset: {df.shape[0]} dòng, {df.shape[1]} cột")

print("\nThông tin DataFrame:")
df.info()

print("\nThống kê mô tả:")
display(df.describe().rename(columns=display_columns_vi))

data_quality = pd.DataFrame(
    {
        "Số giá trị thiếu": df.isna().sum(),
        "Kiểu dữ liệu": df.dtypes.astype(str),
    }
).rename(index=display_columns_vi)
print("\nKiểm tra dữ liệu thiếu và kiểu dữ liệu:")
display(data_quality)

## 6. Tạo X / y

`X` là dữ liệu đầu vào gồm area, rooms và distance. `y` là giá nhà cần dự đoán.

In [ ]:
feature_columns = ["area", "rooms", "distance"]
target_column = "price"

X = df[feature_columns]
y = df[target_column]

print("Các feature đầu vào:", ", ".join(display_columns_vi[column] for column in feature_columns))
print("Target cần dự đoán:", display_columns_vi[target_column])
print(f"Kích thước X: {X.shape}")
print(f"Kích thước y: {y.shape}")

## 7. Chia train/test

Chia dữ liệu thành 80% train và 20% test với random_state = 42 để kết quả có thể tái lập.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=random_seed,
)

print(f"Số mẫu train: {len(X_train)}")
print(f"Số mẫu test: {len(X_test)}")

## 8. Thử nhiều giá trị K

Thử nhiều giá trị K. Mỗi model là một Pipeline gồm StandardScaler và KNeighborsRegressor, vì KNN dựa trên khoảng cách nên cần scale feature.

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15]
metric_rows = []

for k in k_values:
    model = Pipeline(
        [
            ("scaler", StandardScaler()),
            ("knn", KNeighborsRegressor(n_neighbors=k)),
        ]
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    metric_rows.append(
        {
            "k": k,
            "mae": mae,
            "rmse": rmse,
            "r2": r2,
        }
    )

results_df = pd.DataFrame(metric_rows)
print("Đã train và đánh giá xong tất cả giá trị K.")

## 9. Compare metrics

Bang duoi day hien thi MAE, RMSE va R2 cua tung gia tri K. RMSE cang thap thi sai so trung binh cang tot.

In [ ]:
display(results_df.sort_values("k").round(4))

ranked_results_df = results_df.sort_values(["rmse", "mae", "k"]).reset_index(drop=True)
display(ranked_results_df.round(4))

## 10. Select best K

Chon K tot nhat dua tren RMSE thap nhat. Neu co nhieu K gan tuong duong, uu tien K khong qua nho de model on dinh hon.

In [ ]:
min_rmse = results_df["rmse"].min()
near_best_threshold = min_rmse * 1.01
near_best_df = results_df[results_df["rmse"] <= near_best_threshold].copy()

stable_candidates_df = near_best_df[near_best_df["k"] >= 5]
if stable_candidates_df.empty:
    selected_row = ranked_results_df.iloc[0]
else:
    selected_row = stable_candidates_df.sort_values(["rmse", "mae", "k"]).iloc[0]

best_k = int(selected_row["k"])
best_mae = float(selected_row["mae"])
best_rmse = float(selected_row["rmse"])
best_r2 = float(selected_row["r2"])

display(results_df.sort_values("k").round(4))
print(f"Best K: {best_k}")
print(f"Best MAE: {best_mae:.2f} million VND")
print(f"Best RMSE: {best_rmse:.2f} million VND")
print(f"Best R2: {best_r2:.4f}")
print(
    "Selected K explanation: K was selected from the lowest RMSE group; "
    "when scores were within 1%, K >= 5 was preferred to avoid an overly sensitive neighbor setting."
)

## 11. Train final Pipeline

Train final Pipeline voi best_k tren tap train. Pipeline gom StandardScaler va KNeighborsRegressor.

In [ ]:
final_model = Pipeline(
    [
        ("scaler", StandardScaler()),
        ("knn", KNeighborsRegressor(n_neighbors=best_k)),
    ]
)

final_model.fit(X_train, y_train)
print(f"Final model trained with K = {best_k}.")

## 12. Evaluate final model

Danh gia final model tren tap test bang MAE, RMSE va R2.

In [ ]:
final_predictions = final_model.predict(X_test)

final_mae = mean_absolute_error(y_test, final_predictions)
final_rmse = np.sqrt(mean_squared_error(y_test, final_predictions))
final_r2 = r2_score(y_test, final_predictions)

print("Final model metrics")
print(f"MAE : {final_mae:.2f} million VND")
print(f"RMSE: {final_rmse:.2f} million VND")
print(f"R2  : {final_r2:.4f}")

model_summary = pd.DataFrame(
    [
        {"metric": "MAE", "value": final_mae},
        {"metric": "RMSE", "value": final_rmse},
        {"metric": "R2", "value": final_r2},
    ]
)
display(model_summary.round(4))

KNN model characteristics

KNN la instance-based learning nen khong hoc coefficient/intercept nhu Linear Regression. Thay vao do, model du doan dua tren cac diem lang gieng gan nhat trong khong gian feature da scale.

In [ ]:
knn_characteristics = {
    "best_k": best_k,
    "train_samples": len(X_train),
    "feature_count": X_train.shape[1],
    "final_mae": final_mae,
    "final_rmse": final_rmse,
    "final_r2": final_r2,
}

display(pd.DataFrame([knn_characteristics]).round(4))

## 13. Save model

Luu toan bo Pipeline bang joblib vao working directory cua Colab voi ten `knn_house_model.pkl`, sau do kiem tra file ton tai.

In [ ]:
model_path = Path("knn_house_model.pkl")
joblib.dump(final_model, model_path)

if model_path.exists():
    print(f"Model saved successfully: {model_path.resolve()}")
else:
    raise FileNotFoundError(f"Model file was not created: {model_path}")

## 14. Load model

Load lai model bang joblib de dam bao file model co the dung cho buoc inference.

In [ ]:
loaded_model = joblib.load(model_path)
print("Model loaded successfully.")
print(type(loaded_model))

## 15. Test prediction

Tao input mau voi dung thu tu feature: area, rooms, distance. Dung loaded model de predict gia nha.

In [ ]:
sample_input = {
    "area": 85.5,
    "rooms": 3,
    "distance": 5.2,
}

sample_df = pd.DataFrame([sample_input], columns=feature_columns)
predicted_price = float(loaded_model.predict(sample_df)[0])

prediction_result = {
    "area": sample_input["area"],
    "rooms": sample_input["rooms"],
    "distance": sample_input["distance"],
    "predicted_price": round(predicted_price, 2),
    "unit": "million VND",
}

display(pd.DataFrame([prediction_result]))
print(
    f"Predicted price for area={sample_input['area']} m2, "
    f"rooms={sample_input['rooms']}, distance={sample_input['distance']} km: "
    f"{predicted_price:.2f} million VND"
)

## 16. Conclusion

- Bai toan trong notebook nay la KNN Regression cho du doan gia nha.
- StandardScaler can thiet vi KNN dung khoang cach giua cac diem du lieu.
- Notebook da thu nhieu gia tri K: 1, 3, 5, 7, 9, 11 va 15.
- Notebook da chon `best_k` dua tren RMSE thap nhat, co uu tien K on dinh khi ket qua gan tuong duong.
- Final model da duoc danh gia bang MAE, RMSE va R2.
- Pipeline da duoc save/load thanh cong bang joblib.
- Model da predict thu voi input mau va san sang cho Buoc 3: FastAPI Model Server + Ngrok.